## (0) Parameters and Constants

- Service region 20x20 [mile sq.]
- #Demand nodes, $n = 10$ (uniformly distributed)
- Manhattan distance ($l1$)
- Vehicle speed, $v = 20$ [mile/hr], $c_{(i,j)} = \frac{1}{20}(|x_{i}-x_{j}|+|y_{i}-y_{j}|)$ [hr]
- $s_{0}=0.5$ [hr], $s_{1} = 0$ 
- $q_{i} \in \{1,2,3,4,5\}$ [packges/hr]
- Maximum carrying capacity, $Q = 20$ [packages]


In [395]:
no_dock = 0 
no_layer = 1
no_demand_node = 4
truck_capacity = 10
# fixed_setup_time = 0.5
fixed_setup_time = 0
truck_speed = 20

constant_dict = { 'truck_capacity' : truck_capacity,'fixed_setup_time' : fixed_setup_time,
                    'truck_speed': truck_speed,'max_vehicles':6}

edge_plot_config = {'line_width':2.5, 'line_color':'#a26337', 'dash':None, 'name':''}

# Create network components: nodes, arcs
labeling_dict = vis_sol.create_nodes(no_dock,no_demand_node)
docking,customers,depot,depot_s,depot_t,all_depot,nodes,arcs = labeling_dict.values()

In [396]:
## SKIP THIS IF DON'T WANT TO USE NEW RANDOM DISTANCE MAP

# Random distance matrix
distance_matrix, node_position = rand_inst.rand_uniform_dis_mat(nodes,depot[0])
# Random customer demand
_depot_demand = pd.Series([0,0,0], index =all_depot+depot)
customer_demand = rand_inst.rand_cust_demand(customers).append(_depot_demand)
# Visualization 
color_map = vis_sol.create_color_list(nodes,{'O':4, 'c':2})
node_trace = vis_sol.create_node_trace(node_position,color_map,_symbol_dict={'O':'diamond', 'c':'circle'})
print(color_map, node_trace)

{'c_4': 2, 'c_2': 2, 'c_1': 2, 'c_3': 2, 'O': 4} Scatter({
    'marker': {'color': [2, 2, 2, 2, 4],
               'colorscale': [[0.0, 'rgb(0,0,0)'], [0.25, 'rgb(230,0,0)'], [0.5,
                              'rgb(230,210,0)'], [0.75, 'rgb(255,255,255)'], [1.0,
                              'rgb(160,200,255)']],
               'line': {'width': 1},
               'reversescale': True,
               'showscale': False,
               'size': 30,
               'symbol': [circle, circle, circle, circle, diamond]},
    'mode': 'markers+text',
    'name': 'Node',
    'text': [c_4, c_2, c_1, c_3, O],
    'textposition': 'top center',
    'x': array([ 8.95282307,  5.62815573,  2.31172191,  7.93859788, 10.        ]),
    'y': array([ 3.41672767,  4.46824943, 14.98264663,  1.60175355, 10.        ])
})


### Save Instance to pickle

In [372]:
instance_name='DemoInstanceCus{0}_1'.format(no_demand_node);print(instance_name);
instance_data = {'distance_matrix':distance_matrix,
               'node_position':node_position,
               'node_trace':node_trace,
                'customer_demand_df':customer_demand}
# instance_data

CounterExMinTSPInstanceCus5_1


In [378]:
with open('./Instances/%s.pickle'%instance_name,'wb') as f1:
    pk.dump(instance_data,f1)
    print(instance_name+" is saved succuessfully!")

DemoInstanceCus5_1 is saved succuessfully!


### Import instance

In [39]:
instance_name='DemoInstanceCus{0}_1'.format(no_demand_node);print(instance_name);
with open('./Instances/%s.pickle'%instance_name,'rb') as f1:
    r_instance = pk.load(f1)

distance_matrix = r_instance['distance_matrix']
node_trace = r_instance['node_trace']
customer_demand = r_instance['customer_demand_df']

DemoInstanceCus5_1


In [397]:
path1 = {'arcs_list': [('O','c_1'),('c_1','c_2'),('c_2','O')],
                      'config': {'line_width':2.5, 'line_color':'#a26337', 'dash':None, 'name':''}}
path2 = {'arcs_list': [('O','c_2'),('c_2','c_1'),('c_1','O')],
                      'config': {'line_width':2.5, 'line_color':'#a26337', 'dash':None, 'name':''}}
path_arcs_list = [path1,path2]
vis_sol.plot_network(path_arcs_list, node_trace,_cus_dem=customer_demand,_title=instance_name)

### Reset Customer demand value
$q_i = \frac{Q}{d_{0i}}$

In [398]:
testing_cust_dem = dict()
for cus in customers:
#     print(cus,truck_capacity/(2*distance_matrix[('O',cus)]))
#     testing_cust_dem[cus] = truck_capacity/(2*distance_matrix[('O',cus)])
    testing_cust_dem[cus] = 5
testing_cust_dem['O'] = 0
testing_cust_dem
# testing_cust_dem = customer_demand

{'c_1': 5, 'c_2': 5, 'c_3': 5, 'c_4': 5, 'O': 0}

## (1) Initial solution set $\mathbb{R}^{+}$

In [399]:
initializer = init_path.InitialRouteGenerator(no_layer,labeling_dict,
                                              testing_cust_dem,constant_dict,
                                              distance_matrix)

In [400]:
# Generate initial set of feasilble routes
row_labels = ['lr','m']+depot+customers+arcs
x = pd.DataFrame(data =row_labels,columns=['labels'])
initializer.generateRoutes(initRouteDf=x,
                           truck_cap_limit=truck_capacity,
                           max_visited_nodes=3, max_vehicles_per_route=10,mode='all')
x = x.set_index('labels')
# Reformatting
feasibleCols = x.shape[1]
init_route = initializer.generateBasicInitialPatterns(feasibleCols,initRouteDf=x)
init_route.rename(index=lambda x:'route[%d]'%x,inplace=True)

nbInitRoute is set to (#UniqueSequences) * (#MaxVehiclesPerRoute) = 40*10 = 400
progress: 0 / 400
progress: 40 / 400
progress: 80 / 400
progress: 120 / 400
progress: 160 / 400
progress: 200 / 400
progress: 240 / 400
progress: 280 / 400
progress: 320 / 400
progress: 360 / 400
#Feasible Cols: 322
Elapsed-time: 0.33853888511657715


In [401]:
x.head(10)

,route[0],route[1],route[2],route[3],route[4],route[5],route[6],route[7],route[8],route[9],...,route[312],route[313],route[314],route[315],route[316],route[317],route[318],route[319],route[320],route[321]
labels,,,,,,,,,,,,,,,,,,,,,
lr,1.267092,1.267092,1.267092,1.267092,1.267092,1.267092,1.267092,1.267092,1.267092,1.267092,...,1.382161,1.277009,1.277009,1.277009,1.277009,1.277009,1.277009,1.277009,1.277009,1.277009
m,1.000000,2.000000,3.000000,4.000000,5.000000,6.000000,7.000000,8.000000,9.000000,10.000000,...,10.000000,2.000000,3.000000,4.000000,5.000000,6.000000,7.000000,8.000000,9.000000,10.000000
O,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
c_1,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
c_2,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
c_3,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
c_4,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
"(c_4, c_2)",0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
"(c_2, c_4)",0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


### Plot sample of route generated

In [491]:
sample_r = x['route[10]']
sample_arcs = sample_r[sample_r.index.isin(arcs)][sample_r==1]

route_plot_dict = dict(zip(['arcs_list','config'],
                           [sample_arcs.index.to_list(),edge_plot_config]))
# {arcs_list: [('O','c_1'),('c_1','c_2'),('c_2','O')]
#                           config: {'line_width':.., 'line_color':..., 'dash':..., 'name':...}}
# reformatted_arcs = [merge_depot_arcs_var([t],depot,depot_s,depot_t) for t in sample_arcs.index.to_list()]
print('mr:',sample_r['m'], 'lr:',sample_r['lr'])
# vis_sol.plot_network([route_plot_dict],node_trace,_title="Demo Instance",_cus_dem=customer_demand)
vis_sol.plot_network([],node_trace,_title="Demo Instance",_cus_dem=testing_cust_dem)

mr: 1.0 lr: 0.9903594840606995


## (2) Phase I: Minimize #vehicle

In [418]:
phaseI_model = md.phaseIModel(init_route, initializer,
                 distance_matrix,constant_dict)

### (2.1) Build-up model

In [419]:
phaseI_model.buildModel()

Finish generating variables!
Finish generating constrains!
Finish generating objective!


### (2.2) Solve model

In [420]:
phaseI_model.solveModel(np.inf,0.1)

Changed value of parameter PoolSearchMode to 2
   Prev: 0  Min: 0  Max: 2  Default: 0
Parameter TImeLimit unchanged
   Value: inf  Min: 0.0  Max: inf  Default: inf
Changed value of parameter MIPGap to 0.1
   Prev: 0.0001  Min: 0.0  Max: inf  Default: 0.0001
Gurobi Optimizer version 9.1.1 build v9.1.1rc0 (mac64)
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads
Optimize a model with 4 rows, 322 columns and 780 nonzeros
Model fingerprint: 0xaa705440
Variable types: 0 continuous, 322 integer (322 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+00, 1e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+00]
Found heuristic solution: objective 4.0000000
Presolve time: 0.01s
Presolved: 4 rows, 322 columns, 780 nonzeros
Variable types: 0 continuous, 322 integer (322 binary)

Root relaxation: objective 3.000000e+00, 4 iterations, 0.00 seconds

    Nodes    |    Current Node    |     Objective Bounds      |     Wo

In [421]:
sol_obj_phaseI = pd.Series(phaseI_model.model.getVars())
sol_vec_phaseI = pd.DataFrame(index = sol_obj_phaseI.apply(lambda x:x.VarName))
sol_vec_phaseI['value'] = sol_obj_phaseI.apply(lambda x:x.X).values
optimal_routes_phaseI = sol_vec_phaseI.loc[sol_vec_phaseI['value']==1]
optimal_routes_phaseI.index

Index(['route[0]', 'route[272]'], dtype='object')

In [422]:
# list_arcs = formatted_routes_list_I[0]['arcs_list']+formatted_routes_list_I[1]['arcs_list']
# pd.Series(distance_matrix)[list_arcs]/20

In [487]:
import importlib
m1 = sys.modules['visualize_sol']
m2 = sys.modules['random_instance']
m3 = sys.modules['initialize_path']
m4 = sys.modules['model']
importlib.reload(m1)
importlib.reload(m2)
importlib.reload(m3)
importlib.reload(m4)

<module 'model' from '/Users/ravitpichayavet/Documents/GaTechOR/GraduateResearch/CTC_CVRP/Modules/model.py'>

In [492]:
print(x.loc[['m','lr']][optimal_routes_phaseI.index])
formatted_routes_list_I = phaseI_model.getRoute4Plot(_route_name_list=optimal_routes_phaseI.index.to_list(), _colums_df=x,_route_config=edge_plot_config)
vis_sol.plot_network(formatted_routes_list_I,node_trace,_title="Demo Instance",_cus_dem=testing_cust_dem)

        route[0]  route[272]
labels                      
m       1.000000    2.000000
lr      1.267092    1.277009


In [495]:
list_arcs = formatted_routes_list_I[0]['arcs_list']+formatted_routes_list_I[1]['arcs_list']
pd.Series(distance_matrix)[list_arcs],pd.Series(distance_matrix)[list_arcs]/20

(c_1  O      12.670925
 O    c_1    12.670925
 c_3  c_4     2.829199
 c_4  O       7.630449
 c_2  c_3     5.176938
 O    c_2     9.903595
 dtype: float64,
 c_1  O      0.633546
 O    c_1    0.633546
 c_3  c_4    0.141460
 c_4  O      0.381522
 c_2  c_3    0.258847
 O    c_2    0.495180
 dtype: float64)

In [332]:
# import importlib
# m1 = sys.modules['visualize_sol']
# m2 = sys.modules['random_instance']
# m3 = sys.modules['initialize_path']
# m4 = sys.modules['model']
# importlib.reload(m1)
# importlib.reload(m2)
# importlib.reload(m3)
# importlib.reload(m4)

### Solve TSP

In [424]:
initializer_TSP = init_path.InitialRouteGenerator(no_layer,labeling_dict,
                                              testing_cust_dem,constant_dict,
                                              distance_matrix)

In [425]:
# Generate initial set of feasilble routes
row_labels = ['lr','m']+depot+customers+arcs
y = pd.DataFrame(data =row_labels,columns=['labels'])
initializer_TSP.generateRoutes(initRouteDf=y,
                           truck_cap_limit=1000,
                           max_visited_nodes=no_demand_node, max_vehicles_per_route=1,mode='all')
y = y.set_index('labels')
# Reformatting
feasibleCols = y.shape[1]
init_route_TSP = initializer_TSP.generateBasicInitialPatterns(feasibleCols,initRouteDf=y)
init_route_TSP.rename(index=lambda y:'route[%d]'%y,inplace=True)

nbInitRoute is set to (#UniqueSequences) * (#MaxVehiclesPerRoute) = 64*1 = 64
progress: 0 / 64
progress: 32 / 64
#Feasible Cols: 64
Elapsed-time: 0.11521196365356445


In [426]:
# y.head(10)
testing_cust_dem

{'c_1': 5, 'c_2': 5, 'c_3': 5, 'c_4': 5, 'O': 0}

In [427]:
TSP_model = TSPModel(init_route_TSP, initializer_TSP,
                 distance_matrix,constant_dict)
TSP_model.buildModel()
timeLimit=120;GAP=0.01;TSP_model.solveModel()

Finish generating variables!
Finish generating constrains!
Finish generating cost vector!....Elapsed-time: 0.08699393272399902
Finish generating objective!
Changed value of parameter PoolSearchMode to 2
   Prev: 0  Min: 0  Max: 2  Default: 0
Gurobi Optimizer version 9.1.1 build v9.1.1rc0 (mac64)
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads
Optimize a model with 5 rows, 64 columns and 260 nonzeros
Model fingerprint: 0x73902066
Variable types: 0 continuous, 64 integer (64 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [2e+00, 3e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+00]
Found heuristic solution: objective 21.0691712
Presolve removed 4 rows and 40 columns
Presolve time: 0.00s
Presolved: 1 rows, 24 columns, 24 nonzeros
Variable types: 0 continuous, 24 integer (24 binary)

Root relaxation: objective 2.106917e+01, 1 iterations, 0.00 seconds

    Nodes    |    Current Node    |     Objective Bou

In [428]:
sol_obj_TSP = pd.Series(TSP_model.model.getVars())
sol_vec_TSP = pd.DataFrame(index = sol_obj_TSP.apply(lambda x:x.VarName))
sol_vec_TSP['value'] = sol_obj_TSP.apply(lambda x:x.X).values
optimal_routes_TSP = sol_vec_TSP.loc[sol_vec_TSP['value']==1]
optimal_routes_TSP.index

Index(['route[63]'], dtype='object')

In [429]:
total_dem = sum(testing_cust_dem.values())
L_TSP = TSP_model.model.ObjVal/(total_dem/2)
m_TSP = np.ceil(L_TSP*total_dem/constant_dict['truck_capacity'])
print(m_TSP)

5.0


In [430]:
testing_cust_dem

{'c_1': 5, 'c_2': 5, 'c_3': 5, 'c_4': 5, 'O': 0}

In [411]:
ans = y['route[63]']
b = pd.Series(ans[ans==1][7:].index).apply(lambda x: distance_matrix[x]/20)

In [432]:
# cov_hull = pd.Series([('O','c_3'),('c_3','c_4'),('c_4','c_2'),('c_2','c_1'),('c_1','c_5'),('c_5','O')])
# cov_ans = cov_hull.apply(lambda x: distance_matrix[x]/20)

In [365]:
cov_ans.sum()

3.033715910980355

In [370]:
b.sum()

3.033715910980356

In [497]:
print(y.loc[['m','lr']][optimal_routes_TSP.index])
formatted_routes_list_I = TSP_model.getRoute4Plot(_route_name_list=optimal_routes_TSP.index.to_list(), _colums_df=y,_route_config=edge_plot_config)
vis_sol.plot_network(formatted_routes_list_I,node_trace,_title="Demo Instance",_cus_dem=testing_cust_dem)


        route[63]
labels           
m        1.000000
lr       2.106917


In [498]:
list_arcs = formatted_routes_list_I[0]['arcs_list']
pd.Series(distance_matrix)[list_arcs],pd.Series(distance_matrix)[list_arcs]/20

(c_4  c_3     2.829199
 O    c_4     7.630449
 c_2  c_1    13.830831
 c_3  c_2     5.176938
 c_1  O      12.670925
 dtype: float64,
 c_4  c_3    0.141460
 O    c_4    0.381522
 c_2  c_1    0.691542
 c_3  c_2    0.258847
 c_1  O      0.633546
 dtype: float64)

In [273]:
sum(pd.Series(distance_matrix)[list_arcs]/20)

3.609961610182282

In [102]:
class TSPModel:
    def __init__(self, _init_route, _initializer,
                 _distance_matrix,constant_dict,
                 extra_constr=None, _model_name = "PhaseII", _mode=None):
        
        self.init_route = _init_route.copy()
        self.route_coeff = _init_route['PathCoeff'].values
        self.init_routes_df = _initializer.init_routes_df
        
        self.depot = _initializer.depot
        self.depot_s = _initializer.depot_s
        self.depot_t = _initializer.depot_t
        self.all_depot = _initializer.all_depot
        self.customers = _initializer.customers
        self.nodes = _initializer.nodes
        self.arcs = _initializer.arcs
        self.route_cost = []
    

        self.coeff_series = _initializer.init_routes_df['labels']
        self.depot_index = self.coeff_series[self.coeff_series.isin(self.all_depot)].index.values
        self.depot_s_index = self.coeff_series[self.coeff_series.isin(self.depot_s)].index.values
        self.depot_t_index = self.coeff_series[self.coeff_series.isin(self.depot_t)].index.values
        self.customer_index = self.coeff_series[self.coeff_series.isin(self.customers)].index.values
        self.nodes_index = self.coeff_series[self.coeff_series.isin(self.nodes)].index.values
        self.arcs_index = self.coeff_series[self.coeff_series.isin(self.arcs)].index.values
        self.veh_no_index = self.coeff_series.loc[self.coeff_series=='m'].index.values
        self.lr_index = self.coeff_series.loc[self.coeff_series=='lr'].index.values
    
        self.constant_dict = constant_dict
        self.vehicle_capacity = constant_dict['truck_capacity']
        self.fixed_setup_time = constant_dict['fixed_setup_time']
        self.truck_speed = constant_dict['truck_speed']
        self.distance_matrix = _distance_matrix
        self.customer_demand = _initializer.customer_demand
#         self.max_vehicles = constant_dict['max_vehicles']
        self.max_vehicles = 1
        
        self.route_index = pd.Series(self.init_route.index).index.values
        self.model = Model(_model_name)
        if _mode is None: self.mode='multiObjective'
        else: self.mode=_mode

    def buildModel(self):
        self.generateVariables()
        self.generateConstraints()
        self.generateCostOfRoutes()
        self.generateObjective()
        self.model.update()
        
    def generateVariables(self):
        self.route = self.model.addVars(self.route_index, lb=0,
                                       vtype=GRB.BINARY, name='route')
        print('Finish generating variables!')
        
    def generateConstraints(self):          
        const1 = ( quicksum(self.route_coeff[rt][i]*self.route[rt] for rt in self.route_index) == int(1) \
                             for i in self.customer_index )
        self.model.addConstrs( const1,name='customer_coverage' )
        
        const2 = (quicksum(self.route[rt]*(self.route_coeff[rt][self.veh_no_index[0]]) for rt in self.route_index)==self.max_vehicles)
        self.model.addConstr(const2,name='vehicles_usage' )
        print('Finish generating constrains!')
    
    def generateCostOfRoutes(self):
        t1=time.time()
        self.route_cost = self.init_routes_df.set_index('labels').apply(lambda col: self.calculateCostOfRoute(col),axis=0)
        print('Finish generating cost vector!....Elapsed-time:',time.time()-t1)
    def calculateCostOfRoute(self, route):
        visiting_nodes = pd.Series(route[self.customer_index][route==1].index)
        visiting_arcs = pd.Series(route[self.arcs_index][route==1].index)
        next_node = ['STR']
#         print(visiting_arcs)
        route_cost = 0
        qr = visiting_nodes.apply(lambda x: self.customer_demand[x]).sum()
        avg_waiting = qr*route['lr']/(2*route['m'])
        while next_node[0]!=self.depot[0]:
            if next_node[0] == 'STR': next_node.pop(0);selecting_node = self.depot[0] #only for the first arc
            else: selecting_node = next_node.pop(0)
#             print(selecting_node)
            outgoing_arc = visiting_arcs[visiting_arcs.apply(lambda x: x[0]==selecting_node)].to_list()[0]
#             outgoing_arc = outgoing_arc[outgoing_arc>0]
#             print(outgoing_arc)
#             arc = visiting_arcs[visiting_arcs.str.contains(selecting_node+',')].values[0].split(',')
            node_j = outgoing_arc[1]
            next_node.append(node_j)
            qj = self.customer_demand[node_j]
#             formatted_arc = tuple(merge_depot_arcs_var([','.join(arc)],self.depot,
#                                                           self.depot_s,self.depot_t)[0].split(','))
#             traveling_time_carrying_pkg = qr*(self.distance_matrix[formatted_arc])/self.constant_dict['truck_speed']
            traveling_time_carrying_pkg = qr*(self.distance_matrix[outgoing_arc])/self.constant_dict['truck_speed']
#             print(arc, qj, self.distance_matrix[formatted_arc])
#             print(traveling_time_carrying_pkg)
            route_cost+=traveling_time_carrying_pkg
            qr = qr-qj
    
#         print(avg_waiting,route_cost)
#         route_cost += avg_waiting
#         print(route_cost)
        cost_dict = dict(zip(['total_cost','avg_waiting','avg_travel'],[route_cost+avg_waiting,avg_waiting,route_cost]))
        return cost_dict
        
    def generateObjective(self):
        # Minimize the total cost of the used rolls
#         if self.mode=='multiObjective':
#             self.model.setObjective( quicksum(self.route[rt]*(self.route_cost[rt]['total_cost']) for rt in self.route_index) ,
#                                     sense=GRB.MINIMIZE)
#         elif self.mode=='TSPOnly':
        self.model.setObjective( quicksum(self.route[rt]*(self.route_cost[rt]['avg_waiting']) for rt in self.route_index) ,
                                    sense=GRB.MINIMIZE)
#         elif self.mode=='TRPOnly':
#             self.model.setObjective( quicksum(self.route[rt]*(self.route_cost[rt]['avg_travel']) for rt in self.route_index) ,
#                                     sense=GRB.MINIMIZE)
        print('Finish generating objective!')
        
    def solveModel(self, timeLimit = None,GAP=None):
        if timeLimit is not None: self.model.setParam('TImeLimit', timeLimit)
        if GAP is not None: self.model.setParam('MIPGap',GAP)
        self.model.setParam('SolutionNumber',2)
        self.model.setParam(GRB.Param.PoolSearchMode, 2)
        self.model.optimize()
        
    ##RELAXATION
    def solveRelaxedModel(self):
        #Relax integer variables to continous variables
        self.relaxedModel = self.model.relax()
        var_ss = pd.Series(self.relaxedModel.getVars())
        var_ss.apply(lambda x: x.setAttr('ub',GRB.INFINITY))
        self.relaxedModel.optimize()
        
    def getRelaxSolution(self):
        a = pd.Series(self.relaxedModel.getAttr('X'))
        return a[a>0]

    def getDuals(self):
        return self.relaxedModel.getAttr('Pi',self.model.getConstrs())

    def getRoute4Plot(self, _route_name_list, _colums_df,_route_config):
        reformatted_arcs=[]
        for idx in _route_name_list:
            sample_r = _colums_df.loc[:][idx]
            sample_arcs = sample_r[sample_r.index.isin(self.arcs)][sample_r==1]
            route_plot_dict = dict(zip(['arcs_list','config'],
                           [sample_arcs.index.to_list(),_route_config]))
            reformatted_arcs += [route_plot_dict]
        return reformatted_arcs
    

## (3) Phase II: Cycle time VRP
(Set partitioning)

In [79]:
phaseII_model = md.phaseIIModel(init_route, initializer,
                 distance_matrix,constant_dict,_mode='TSPOnly')

### (3.1) Build-up model

In [80]:
phaseII_model.buildModel()

Finish generating variables!
Finish generating constrains!
Finish generating cost vector!....Elapsed-time: 0.16881370544433594
Finish generating objective!


### (3.2) Solve model

In [81]:
timeLimit=120;GAP=0.01;phaseII_model.solveModel()

Changed value of parameter PoolSearchMode to 2
   Prev: 0  Min: 0  Max: 2  Default: 0
Gurobi Optimizer version 9.1.1 build v9.1.1rc0 (mac64)
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads
Optimize a model with 6 rows, 150 columns and 420 nonzeros
Model fingerprint: 0xf52dc8a1
Variable types: 0 continuous, 150 integer (150 binary)
Coefficient statistics:
  Matrix range     [1e+00, 6e+00]
  Objective range  [1e-01, 1e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 6e+00]
Presolve removed 0 rows and 25 columns
Presolve time: 0.01s
Presolved: 6 rows, 125 columns, 341 nonzeros
Variable types: 0 continuous, 125 integer (125 binary)
Found heuristic solution: objective 7.0391267

Root relaxation: objective 4.753437e+00, 31 iterations, 0.00 seconds

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

     0     0    4.75344    0    5    7.03913    4.7

In [82]:
sol_obj_phaseII = pd.Series(phaseII_model.model.getVars())
sol_vec_phaseII = pd.DataFrame(index = sol_obj_phaseII.apply(lambda x:x.VarName))
sol_vec_phaseII['value'] = sol_obj_phaseII.apply(lambda x:x.X).values
optimal_routes_phaseII = sol_vec_phaseII.loc[sol_vec_phaseII['value']==1]
optimal_routes_phaseII.index

Index(['route[19]', 'route[50]', 'route[102]'], dtype='object')

In [83]:
print(x.loc[['m','lr']][optimal_routes_phaseII.index])
formatted_routes_list_II =  phaseII_model.getRoute4Plot(optimal_routes_phaseII.index.to_list(),
                                                        x,edge_plot_config)
vis_sol.plot_network(formatted_routes_list_II,node_trace,_title="Demo Instance",_cus_dem=customer_demand)

        route[19]  route[50]  route[102]
labels                                  
m        2.000000   3.000000     1.00000
lr       2.017318   2.120752     1.63051
